In [1]:
!pip install yfinance==0.2.66

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

ticker = "XOM"

end_date = datetime.now()
start_date = end_date - timedelta(days=7)   # Yahoo only serves 1m data for ~trailing 7 days

all_data = pd.DataFrame()
current_end = end_date
max_retries = 5
base_delay = 10
chunk_delay = 5
chunk_days = 1

def is_weekend_only(start, end):
    d = start
    while d < end:
        if d.weekday() < 5:  # Mon-Fri
            return False
        d += timedelta(days=1)
    return True

while current_end > start_date:
    current_start = max(current_end - timedelta(days=chunk_days), start_date)

    if is_weekend_only(current_start, current_end):
        logger.info(f"Skipping {current_start} to {current_end} — weekend, market closed.")
        current_end = current_start
        continue

    logger.info(f"Fetching data from {current_start} to {current_end}")

    for attempt in range(max_retries):
        try:
            chunk = yf.download(ticker, start=current_start, end=current_end, interval="1m", auto_adjust=False)

            if not chunk.empty:
                all_data = pd.concat([all_data, chunk])
                logger.info(f"Successfully fetched data for {current_start} to {current_end}")
                break
            else:
                logger.warning(f"Empty data for {current_start} to {current_end}, retrying...")

        except Exception as e:
            logger.error(f"Error on attempt {attempt + 1}/{max_retries}: {e}")

        if attempt < max_retries - 1:
            delay = base_delay * (2 ** attempt)
            logger.info(f"Waiting {delay} seconds before retrying...")
            time.sleep(delay)
        else:
            logger.error(f"Max retries reached for {current_start} to {current_end}, skipping...")

        time.sleep(2)

    time.sleep(chunk_delay)
    current_end = current_start

all_data = all_data[~all_data.index.duplicated(keep='first')]
all_data = all_data.sort_index()

print(f"Total rows fetched: {len(all_data)}")
print(all_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Total rows fetched: 1948
Price                       Adj Close       Close        High         Low  \
Ticker                            XOM         XOM         XOM         XOM   
Datetime                                                                    
2026-09-14 13:30:00+00:00  168.750000  168.750000  168.910004  168.500000   
2026-09-14 13:31:00+00:00  168.744995  168.744995  168.839996  168.630005   
2026-09-14 13:32:00+00:00  168.847900  168.847900  168.960007  168.220001   
2026-09-14 13:33:00+00:00  169.225006  169.225006  169.300003  168.664993   
2026-09-14 13:34:00+00:00  168.550003  168.550003  169.449997  168.550003   

Price                            Open  Volume  
Ticker                            XOM     XOM  
Datetime                                       
2026-09-14 13:30:00+00:00  168.699997  136579  
2026-09-14 13:31:00+00:00  168.800003    8457  
2026-09-14 13:32:00+00:00  168.500000  337037  
2026-09-14 13:33:00+00:00  168.789993  230299  
2026-09-14 13:34:00+00

In [2]:
if isinstance(all_data.columns, pd.MultiIndex):
    all_data.columns = all_data.columns.get_level_values(0)

all_data = all_data.reset_index()
all_data = all_data.rename(columns={'Datetime': 'Timestamp'})
all_data.columns.name = None

cols = ['Timestamp'] + [c for c in all_data.columns if c != 'Timestamp']
all_data = all_data[cols]

filename = f"{ticker}.csv"
all_data.to_csv(filename, index=False)
logger.info(f"Data saved to {filename} (flattened, {len(all_data)} rows)")

print(all_data.head())
print(all_data.dtypes)

                  Timestamp   Adj Close       Close        High         Low  \
0 2026-09-14 13:30:00+00:00  168.750000  168.750000  168.910004  168.500000   
1 2026-09-14 13:31:00+00:00  168.744995  168.744995  168.839996  168.630005   
2 2026-09-14 13:32:00+00:00  168.847900  168.847900  168.960007  168.220001   
3 2026-09-14 13:33:00+00:00  169.225006  169.225006  169.300003  168.664993   
4 2026-09-14 13:34:00+00:00  168.550003  168.550003  169.449997  168.550003   

         Open  Volume  
0  168.699997  136579  
1  168.800003    8457  
2  168.500000  337037  
3  168.789993  230299  
4  169.250000  107213  
Timestamp    datetime64[ns, UTC]
Adj Close                float64
Close                    float64
High                     float64
Low                      float64
Open                     float64
Volume                     int64
dtype: object


In [3]:
from google.colab import files
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>